In [11]:
import sys
from pathlib import Path

project_root = Path.cwd().parents[0]  # importing functions from other folders

sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import os
from _data.data_utils import read_in
from _fitting.fitting_utils import abbrev_stat
from _fitting.model_utils_smooth import *
# from _fitting.fitting_utils import hist_plot, CI_plot, CI_plot_alt, CI_plot_both, plot_posteriors_side_by_side, plot_spline_Bknots
import pymc as pm
import pymc.math as pmm
import arviz as az
from patsy import dmatrix
import nutpie
import time
from IPython.display import display
from pymc.variational.callbacks import CheckParametersConvergence
import io
import base64
import re
import pytensor.tensor as pt
from pytensor.gradient import disconnected_grad
from scipy.sparse.linalg import eigsh
from matplotlib.colors import Normalize, TwoSlopeNorm, to_rgba

az.style.use("arviz-darkgrid")

if '___laptop' in os.listdir('../'):

    # laptop folder
    folder = "../../_data/p-dengue/"
elif '___server' in os.listdir('../'):
    # server folder
    folder = "../../../../../data/lucaratzinger_data/p_dengue/"

%matplotlib inline
import seaborn as sns

import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

###
from _fitting.model_utils import data_settings_to_name
###

In [12]:
from _fitting.model_utils_smooth import build_upars

In [13]:
idata_file = os.path.join(folder, "model_fits", "a2_201601_201912[22_smooth_exact_simple_greedy_1_region_mm_full]", "idata", "idata_[[t2m_max_p(1)(p=2.5, 5)]].nc")
idata = az.from_netcdf(idata_file)

In [14]:
uparam_names = ['alpha', 'intercept', 'beta_u',
                'w(t2m_max_pop_weighted(1))[0]',
                'w(t2m_max_pop_weighted(1))[1]',
                'w(t2m_max_pop_weighted(1))[2]',
                'w(t2m_max_pop_weighted(1))[3]',
                'w(t2m_max_pop_weighted(1))[4]',
                'w(t2m_max_pop_weighted(1))[5]',
                'w(t2m_max_pop_weighted(1))[6]',
                'w(t2m_max_pop_weighted(1))[7]',]

log_transformed = ['alpha']

In [15]:
uparam_names

['alpha',
 'intercept',
 'beta_u',
 'w(t2m_max_pop_weighted(1))[0]',
 'w(t2m_max_pop_weighted(1))[1]',
 'w(t2m_max_pop_weighted(1))[2]',
 'w(t2m_max_pop_weighted(1))[3]',
 'w(t2m_max_pop_weighted(1))[4]',
 'w(t2m_max_pop_weighted(1))[5]',
 'w(t2m_max_pop_weighted(1))[6]',
 'w(t2m_max_pop_weighted(1))[7]']

In [16]:
bu = build_upars(idata, uparam_names, log_transformed)

In [17]:
bu

<xarray.DataArray 'alpha' (chain: 4, draw: 4000, uparam: 11)> Size: 1MB
array([[[-9.19063528e-01, -1.01409257e+01,  9.91232949e-02, ...,
         -1.76561983e+00,  3.65805169e+00, -8.43782031e+00],
        [-9.27968429e-01, -1.01347358e+01,  1.24697635e-01, ...,
         -1.91655401e+00,  2.66155468e+00, -7.98312315e+00],
        [-9.13762464e-01, -1.01369670e+01,  1.05002447e-01, ...,
         -5.43473381e+00,  1.01072687e+01, -3.34481341e+00],
        ...,
        [-9.06322612e-01, -1.01527446e+01,  9.52554301e-02, ...,
          1.95677375e+00,  4.38192989e+00, -6.73926879e+00],
        [-9.34862504e-01, -1.01540110e+01,  8.52614617e-02, ...,
         -3.64882904e+00,  6.24404399e+00, -6.45029013e+00],
        [-8.98668056e-01, -1.01239140e+01,  1.00135332e-01, ...,
         -2.60447520e+00,  1.05154806e+01, -5.99537691e+00]],

       [[-9.31309432e-01, -1.01504403e+01,  9.10564688e-02, ...,
         -5.59904479e+00,  2.17958737e+01, -1.10407084e+01],
        [-9.02015856e-01, -1.01376096e+01,  9.84695956e-02, ...,
          1.13504936e-01, -1.19675470e+01, -3.14909321e+00],
        [-9.32868218e-01, -1.01223147e+01,  9.11597250e-02, ...,
         -8.64551682e-03,  1.10608713e+01, -1.06574623e+01],
...
         -6.90106789e+00,  1.22290230e+01, -5.97540743e+00],
        [-9.31698295e-01, -1.01597171e+01,  8.57086336e-02, ...,
         -3.49716995e+00,  1.25359571e+00, -1.14421429e+01],
        [-9.24693730e-01, -1.01821519e+01,  9.22626231e-02, ...,
         -5.71972777e+00,  4.88235609e+00, -1.31310883e+01]],

       [[-8.94137679e-01, -1.01539250e+01,  9.80385396e-02, ...,
         -1.67611476e+00, -7.13126762e+00, -6.24832188e+00],
        [-9.30270745e-01, -1.01514062e+01,  1.02630628e-01, ...,
         -6.58330931e+00,  1.20699342e+01, -5.08466753e+00],
        [-9.01309369e-01, -1.01519032e+01,  9.52772196e-02, ...,
         -3.02079629e+00,  2.01537273e+00, -1.16489797e+01],
        ...,
        [-9.04190569e-01, -1.01241479e+01,  1.02906095e-01, ...,
          1.00515986e+00, -4.21268594e+00, -1.08133779e+00],
        [-9.25195380e-01, -1.01443672e+01,  1.06781556e-01, ...,
         -9.67187366e-01,  6.98903548e+00, -1.13571440e+01],
        [-9.11455943e-01, -1.01157940e+01,  8.66370747e-02, ...,
         -7.64934314e-01, -1.63038123e+00, -6.05511283e+00]]],
      shape=(4, 4000, 11))
Coordinates:
  * chain    (chain) int64 32B 0 1 2 3
  * draw     (draw) int64 32kB 0 1 2 3 4 5 6 ... 3994 3995 3996 3997 3998 3999
  * uparam   (uparam) object 88B 'alpha' ... 'w(t2m_max_pop_weighted(1))[7]'

In [18]:
data_settings = {'admin':2, 'max_lag':6, 'start_year':2016, 'start_month':1, 'end_year':2019, 'end_month':12,
                 'select_admin1_regions':None}
data_name = data_settings_to_name(data_settings)

fitting_task = '22_smooth_exact_simple_greedy_1_region_mm_full'

################################################################################


################################################################################
_data = read_in(folder, **data_settings, standardise=True, dropna=True, celsius=True, tp_log=True)
statistics = _data.columns.tolist()
# that start with t2m, rh, tp
# statistics = [name for name in statistics if name.startswith(('t2m','rh','tp'))]
statistics = [name for name in statistics if name.startswith(('t2m_max'))]
# that contains 'pop_weighted'
statistics = [name for name in statistics if 'pop_weighted' in name]
# if it contains 'tp' then it should contain 'log('
statistics = [name for name in statistics if not (name.startswith('tp') and 'log(' not in name)]
lags = [0, 1, 2, 3, 4, 5, 6]
statistics = [stat for stat in statistics if any(f"({lag})" in stat for lag in lags)]

print(len(statistics))
print(statistics)
################################################################################
selected_complete_simple = { # adjusted for pareto k and sampling
    'rh_mean_pop_weighted(0)': [(2.5, 5)],
    'rh_mean_pop_weighted(1)': [(2.5, 5)],
    'rh_mean_pop_weighted(2)': [(2.5, 5)],
    'rh_mean_pop_weighted(3)': [(2.5, 5)],
    'rh_mean_pop_weighted(4)': [(2.5, 5)],
    'rh_mean_pop_weighted(5)': [(2.5, 5)],
    'rh_mean_pop_weighted(6)': [(2.5, 5)],
    ###
    't2m_max_pop_weighted(0)': [(2.5, 20)],
    't2m_max_pop_weighted(1)': [(2.5, 20)],
    't2m_max_pop_weighted(2)': [(2.5, 20)],
    't2m_max_pop_weighted(3)': [(2.5, 20)],
    't2m_max_pop_weighted(4)': [(2.5, 20)],
    't2m_max_pop_weighted(5)': [(2.5, 20)],
    't2m_max_pop_weighted(6)': [(2.5, 20)],
    ###
    't2m_mean_pop_weighted(0)': [(5.0, 10)],
    't2m_mean_pop_weighted(1)': [(5.0, 10)],
    't2m_mean_pop_weighted(2)': [(5.0, 10)],
    't2m_mean_pop_weighted(3)': [(5.0, 10)],
    't2m_mean_pop_weighted(4)': [(5.0, 10)],
    't2m_mean_pop_weighted(5)': [(5.0, 10)],
    't2m_mean_pop_weighted(6)': [(5.0, 10)],
    ###
    't2m_min_pop_weighted(0)': [(2.5, 20)],
    't2m_min_pop_weighted(1)': [(2.5, 20)],
    't2m_min_pop_weighted(2)': [(2.5, 20)],
    't2m_min_pop_weighted(3)': [(2.5, 20)],
    't2m_min_pop_weighted(4)': [(2.5, 20)],
    't2m_min_pop_weighted(5)': [(2.5, 20)],
    't2m_min_pop_weighted(6)': [(2.5, 20)],
    ###
    'tp_24hmax_pop_weighted_log(0)': [(2.5, 5)],
    'tp_24hmax_pop_weighted_log(1)': [(2.5, 5)],
    'tp_24hmax_pop_weighted_log(2)': [(2.5, 5)],
    'tp_24hmax_pop_weighted_log(3)': [(2.5, 5)],
    'tp_24hmax_pop_weighted_log(4)': [(2.5, 5)],
    'tp_24hmax_pop_weighted_log(5)': [(2.5, 5)],
    'tp_24hmax_pop_weighted_log(6)': [(2.5, 5)],
    ###
    'tp_24hmean_pop_weighted_log(0)': [(2.5, 5)],
    'tp_24hmean_pop_weighted_log(1)': [(2.5, 5)],
    'tp_24hmean_pop_weighted_log(2)': [(2.5, 5)],
    'tp_24hmean_pop_weighted_log(3)': [(2.5, 5)],
    'tp_24hmean_pop_weighted_log(4)': [(2.5, 5)],
    'tp_24hmean_pop_weighted_log(5)': [(2.5, 5)],
    'tp_24hmean_pop_weighted_log(6)': [(2.5, 5)],
}

selected_complete_simple = {s:[(2.5, 5)] for s in statistics}
p = {s: vals[0][0] for s, vals in selected_complete_simple.items()}
num_knots = {s: vals[0][1] for s, vals in selected_complete_simple.items()}

# greedy_0
# s1_list = None

# greedy_1
s1_list = []
s1_list.append([])

# greedy_2
# s1_list.append(['tp_24hmean_pop_weighted_log(1)']) #1
# s1_list.append(['rh_mean_pop_weighted(1)']) #2
# s1_list.append(['rh_mean_pop_weighted(0)']) #3

# greedy_3
# s1_list.append(['tp_24hmean_pop_weighted_log(1)', 't2m_max_pop_weighted(4)']) #1
# s1_list.append(['tp_24hmean_pop_weighted_log(1)', 't2m_mean_pop_weighted(4)']) #2
# s1_list.append(['rh_mean_pop_weighted(1)', 'tp_24hmean_pop_weighted_log(5)']) #3

# greedy_4
# s1_list.append(['rh_mean_pop_weighted(1)', 'tp_24hmean_pop_weighted_log(5)', 't2m_max_pop_weighted(6)']) #1
# s1_list.append(['rh_mean_pop_weighted(1)', 'tp_24hmean_pop_weighted_log(5)', 't2m_mean_pop_weighted(3)']) #2
# s1_list.append(['rh_mean_pop_weighted(1)', 'tp_24hmean_pop_weighted_log(5)', 't2m_mean_pop_weighted(1)']) #3

# greedy_5
# s1_list.append(['rh_mean_pop_weighted(1)', 'tp_24hmean_pop_weighted_log(5)', 't2m_max_pop_weighted(6)', 'tp_24hmax_pop_weighted_log(2)']) #1
# s1_list.append(['rh_mean_pop_weighted(1)', 'tp_24hmean_pop_weighted_log(5)', 't2m_mean_pop_weighted(1)', 'tp_24hmean_pop_weighted_log(2)']) #2
# s1_list.append(['rh_mean_pop_weighted(1)', 'tp_24hmean_pop_weighted_log(5)', 't2m_mean_pop_weighted(3)', 'tp_24hmean_pop_weighted_log(2)']) #3

# greedy_6
# s1_list.append(['rh_mean_pop_weighted(1)', 'tp_24hmean_pop_weighted_log(5)', 't2m_max_pop_weighted(6)', 'tp_24hmax_pop_weighted_log(2)', 'tp_24hmean_pop_weighted_log(6)']) #1
# s1_list.append(['rh_mean_pop_weighted(1)', 'tp_24hmean_pop_weighted_log(5)', 't2m_mean_pop_weighted(1)', 'tp_24hmean_pop_weighted_log(2)', 't2m_max_pop_weighted(5)']) #2
# s1_list.append(['rh_mean_pop_weighted(1)', 'tp_24hmean_pop_weighted_log(5)', 't2m_max_pop_weighted(6)', 'tp_24hmax_pop_weighted_log(2)', 'tp_24hmean_pop_weighted_log(0)']) #3


# s1_list = [['t2m_mean_pop_weighted(1)']]
if __name__ == "__main__":
    # Build model dictionary
    model_dict = {}
    if s1_list is not None:
        for s1 in s1_list:
            statistics_s1 = statistics.copy()
            for s in s1:
                statistics_s1.remove(s)

            for stat_name in statistics_s1:
                stat_names = s1 + [stat_name]
                settings = {
                    'alpha_type': 'exponential', 'alpha_parameters': {'lam': 0.5},
                    'intercept_type': 'normal', 'intercept_parameters': {'mu': -10.0, 'sigma': 1.0},
                    'beta_u_type': 'normal', 'beta_u_parameters': {'mu': 0, 'sigma': 1.0},
                    'link': None, 'link_stat_name': None, 'link_type': None,
                    'b1_type': None, 'b1_parameters': None,
                    'c_type': None, 'c_parameters': None,
                    'stat_names': stat_names, 'num_knots': num_knots, 'knot_type': 'equispaced', 'degree': 3,
                    'spline_implementation': 'svd', 'spline_type': 'halfnormal', 'spline_parameters': {'sigma_w_sigma': 10.0},
                    'penalty_order': 2, 'penalty_type': 'halfnormal', 'penalty_parameters': {'p': p}, 'penalty_std': False,
                    'cutoff': None,
                    'exclude': ['intercept', 'beta_u', 'alpha', 'zi_b1', 'zi_c'],
                    'surveillance_name': None,
                    'urbanisation_name': 'urbanisation_pop_weighted_std'}
                
                model_name = build_model_name_mult_stat(settings['alpha_type'], settings['alpha_parameters'],
                                settings['intercept_type'], settings['intercept_parameters'],
                                settings['link'], settings['link_stat_name'], settings['link_type'],
                                settings['b1_type'], settings['b1_parameters'],
                                settings['c_type'], settings['c_parameters'],
                                settings['stat_names'], settings['num_knots'], settings['knot_type'], settings['degree'],
                                settings['spline_implementation'], settings['spline_type'], settings['spline_parameters'],
                                settings['penalty_order'], settings['penalty_type'], settings['penalty_parameters'], settings['penalty_std'],
                                settings['cutoff'], settings['beta_u_type'], settings['beta_u_parameters'],
                                exclude=settings['exclude'],
                                surveillance_name=settings['surveillance_name'],
                                urbanisation_name=settings['urbanisation_name'])
                model_dict[model_name] = settings
    else:
        print("Fitting NB log_population + intercept + urbanisation model")
        stat_names=None
        settings = {
            'alpha_type': 'exponential', 'alpha_parameters': {'lam': 0.5},
            'intercept_type': 'normal', 'intercept_parameters': {'mu': -10.0, 'sigma': 1.0},
            'beta_u_type': 'normal', 'beta_u_parameters': {'mu': 0, 'sigma': 1.0},
            'link': None, 'link_stat_name': None, 'link_type': None,
            'b1_type': None, 'b1_parameters': None,
            'c_type': None, 'c_parameters': None,
            'stat_names': stat_names, 'num_knots': num_knots, 'knot_type': 'equispaced', 'degree': 3,
            'spline_implementation': 'svd', 'spline_type': 'halfnormal', 'spline_parameters': {'sigma_w_sigma': 10.0},
            'penalty_order': 2, 'penalty_type': 'halfnormal', 'penalty_parameters': {'p': p}, 'penalty_std': False,
            'cutoff': None,
            'exclude': ['intercept', 'beta_u', 'alpha', 'zi_b1', 'zi_c'],
            'surveillance_name': None,
            'urbanisation_name': 'urbanisation_pop_weighted_std'}
        
        model_name = build_model_name_mult_stat(settings['alpha_type'], settings['alpha_parameters'],
                        settings['intercept_type'], settings['intercept_parameters'],
                        settings['link'], settings['link_stat_name'], settings['link_type'],
                        settings['b1_type'], settings['b1_parameters'],
                        settings['c_type'], settings['c_parameters'],
                        settings['stat_names'], settings['num_knots'], settings['knot_type'], settings['degree'],
                        settings['spline_implementation'], settings['spline_type'], settings['spline_parameters'],
                        settings['penalty_order'], settings['penalty_type'], settings['penalty_parameters'], settings['penalty_std'],
                        settings['cutoff'], settings['beta_u_type'], settings['beta_u_parameters'],
                        exclude=settings['exclude'],
                        surveillance_name=settings['surveillance_name'],
                        urbanisation_name=settings['urbanisation_name'])
        model_dict[model_name] = settings
        
    # Create tasks list
    tasks = list(model_dict.items())

7
['t2m_max_pop_weighted(0)', 't2m_max_pop_weighted(1)', 't2m_max_pop_weighted(2)', 't2m_max_pop_weighted(3)', 't2m_max_pop_weighted(4)', 't2m_max_pop_weighted(5)', 't2m_max_pop_weighted(6)']


In [19]:
from _fitting.model_utils_smooth import build_model_exact_mult_stat_mm

In [20]:
model_settings = model_dict['[t2m_max_p(1)(p=2.5, 5)]']
model, m, B, V_r_dict, knot_list, log_prob_upars_fn, log_lik_i_upars_fn, uparam_names = build_model_exact_mult_stat_mm(_data.copy(), **model_settings)
model_name = m

In [10]:
# sum log_lik over all observations - this is what PyMC computes
total_ll = idata.log_likelihood['y_obs'].sum(dim='y_obs_dim_0')
print("total ll sample:", total_ll.values[:2, :5])

# our log_prob
lp = log_prob_upars_fn(bu)
print("log_prob sample:", lp.values[:2, :5])

# difference should be the log_prior contribution
diff = lp.values - total_ll.values
print("diff (should be log_prior):", diff[:2, :5])
print("diff mean:", diff.mean())
print("diff std:", diff.std())  # should be near 0 if only a constant prior

KeyboardInterrupt: 

In [21]:
print(bu.dims)   # should be ('chain', 'draw', 'uparam')
print(bu.shape)

('chain', 'draw', 'uparam')
(4, 4000, 11)


In [22]:
log_prob_upars_fn(bu)

sum+total:      6.210s


<xarray.DataArray (chain: 4, draw: 4000)> Size: 128kB
array([[-82682.12475033, -82689.52374372, -82686.18350727, ...,
        -82683.51990796, -82684.18796464, -82684.82109899],
       [-82686.7109989 , -82685.86426459, -82685.16423599, ...,
        -82687.09383489, -82687.92130018, -82688.43044699],
       [-82684.22046285, -82683.41538649, -82682.97921549, ...,
        -82689.19335868, -82686.74037002, -82684.13440914],
       [-82686.62804904, -82684.08683663, -82684.4586997 , ...,
        -82684.87552555, -82683.6122841 , -82684.07644038]],
      shape=(4, 4000))
Dimensions without coordinates: chain, draw

In [12]:
log_lik_i_upars_fn(bu, 0)

<xarray.DataArray (chain: 4, draw: 4000)> Size: 128kB
array([[-3.19924754, -3.20063026, -3.19206811, ..., -3.19749454,
        -3.20985204, -3.19737791],
       [-3.2063187 , -3.19165659, -3.20583154, ..., -3.20018866,
        -3.20486554, -3.20488301],
       [-3.20348256, -3.19548885, -3.19381311, ..., -3.19589095,
        -3.20982438, -3.20471683],
       [-3.19607906, -3.20358023, -3.19781781, ..., -3.19660056,
        -3.19503782, -3.20140689]], shape=(4, 4000))
Dimensions without coordinates: chain, draw

In [ ]:
idata.log_likelihood['y_obs'].isel(y_obs_dim_0=0).values

array([[-3.19924754, -3.20063026, -3.19206811, ..., -3.19749454,
        -3.20985204, -3.19737791],
       [-3.2063187 , -3.19165659, -3.20583154, ..., -3.20018866,
        -3.20486554, -3.20488301],
       [-3.20348256, -3.19548885, -3.19381311, ..., -3.19589095,
        -3.20982438, -3.20471683],
       [-3.19607906, -3.20358023, -3.19781781, ..., -3.19660056,
        -3.19503782, -3.20140689]], shape=(4, 4000))